# 일본어 전처리

- 대응 코드: `preprocess_jp.py`
- 문서 1건의 정제·문장 분리·차단 맥락과 키워드 매칭을 확인한 뒤 표본 실행.
- 비공개 원문은 노트북 출력에 저장하지 않음.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

# 작업 폴더가 저장소 루트이거나 노트북 폴더여도 analysis/를 찾는다.
candidates = [Path.cwd(), *Path.cwd().parents]
analysis_root = next((candidate for base in candidates
                      for candidate in (base, base / "code" / "analysis", base / "ESG_comp" / "code" / "analysis")
                      if (candidate / "common" / "config.py").is_file()), None)
if analysis_root is None:
    raise FileNotFoundError("ESG_comp/code/analysis 폴더에서 노트북을 열어 주세요.")
if not os.environ.get("ESG_RESEARCH_ROOT"):
    raise RuntimeError("비공개 입력 위치를 ESG_RESEARCH_ROOT 환경변수로 먼저 지정하세요.")
sys.path.insert(0, str(analysis_root))
from common import config as cfg
from common.io_common import load_enriched


def show_csv(path, rows=10, columns=None):
    path = Path(path)
    if not path.is_file():
        print(f"저장 결과 없음: {path}\n입력 자료와 앞 단계를 확인한 뒤 실행하세요.")
        return None
    frame = pd.read_csv(path)
    display(frame[columns].head(rows) if columns else frame.head(rows))
    print(f"{path.name}: {len(frame):,}행")
    return frame


In [ ]:
SAMPLE_DOC = 0  # 확인할 일본어 문서 순번
LIMIT_DOCS = 20
RUN_SAMPLE = False
SAVE_SAMPLE = False

In [ ]:
from stage1_preprocess import preprocess_jp as step
from common import io_common, mention_common
if not cfg.FINAL_DATASET_XLSX.is_file() or not cfg.REVIEW_XLSX.is_file():
    raise FileNotFoundError("최종 데이터셋과 수동검토 파일 위치를 확인하세요: ESG_RESEARCH_ROOT/comp/0_crawling/phase5_page_type/")
source = io_common.load_scoped_dataset()
source = source[source["lang"] == "jp"].reset_index(drop=True)
print(f"일본어 입력 문서: {len(source):,}건")
display(source[[c for c in ("entity_id", "page_type_reviewed", "target_field_primary") if c in source]].head(10))

In [ ]:
if not 0 <= SAMPLE_DOC < len(source):
    raise IndexError("SAMPLE_DOC을 일본어 문서 범위로 바꾸세요.")
cleaned = step.clean_text_ja(source.loc[SAMPLE_DOC, "full_text"])
sentences = step.split_sentences_ja(cleaned)
masked = [step._mask_blocked_context(sentence) for sentence in sentences]
sample_matches = mention_common.extract_doc_mentions("sample", masked, step.ALIAS_POOL_JA)
print(f"정제 후 {len(cleaned):,}자 · 문장 {len(sentences):,}개 · 매칭 {len(sample_matches):,}건")
display(pd.DataFrame(sample_matches).head(10))

In [ ]:
if RUN_SAMPLE:
    docs, mentions = step.run(limit_docs=LIMIT_DOCS)
    display(docs[[c for c in ("entity_id", "cleaned_text_len", "page_type_reviewed") if c in docs]].head(10))
    display(mentions.head(10))
    if SAVE_SAMPLE:
        sample_dir = cfg.OUTPUT_DIR / "notebook_samples"
        sample_dir.mkdir(parents=True, exist_ok=True)
        docs.drop(columns=["full_text"]).to_csv(sample_dir / "docs_jp_sample.csv", index=False, encoding="utf-8-sig")
        mentions.to_csv(sample_dir / "mentions_jp_sample.csv", index=False, encoding="utf-8-sig")
else:
    print("표본 전처리 실행은 RUN_SAMPLE=True로 선택하세요.")

In [ ]:
show_csv(cfg.OUTPUT_DIR / "docs_jp.csv")
show_csv(cfg.OUTPUT_DIR / "mentions_jp_raw.csv")